# Tahap 5 - Uji Sensitivitas & Signifikansi

**Tujuan:** menguji apakah temuan Tahap 4 kuat atau hanya kebetulan sampel / pilihan metode.

Prinsipnya: **sebelum mengklaim sebuah temuan, coba jebak agar temuan itu gagal.** Kalau tetap bertahan, barulah temuan itu layak masuk laporan. Semua angka di bawah dihitung dari `outputs/tiktok_clean.csv`, bukan dari Tahap 4 yang dihafal.

Yang diuji:
1. Presisi median (bootstrap 2.000 kali, seed 42).
2. Apakah perbedaan like rate antar durasi benar-benar ada (Kruskal-Wallis + Mann-Whitney).
3. Apakah kesimpulan berubah kalau 8 video paling viral dibuang.
4. Apakah daftar "item di video interaksi tinggi" berubah kalau ambangnya diganti.
5. Apakah efek durasi hanya efek follower atau tahun posting.


In [ ]:
# Sel 1: Siapkan lingkungan, data, dan fungsi bantu
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

BASE = '/content'
sns.set_theme(style='whitegrid', context='notebook')
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3', '#937860']
sns.set_palette(PALETTE)
plt.rcParams.update({'figure.figsize': (10, 6), 'axes.titlesize': 14,
                     'axes.titleweight': 'bold', 'axes.labelsize': 11})
RNG = np.random.default_rng(42)

df = pd.read_csv(os.path.join(BASE, 'outputs', 'tiktok_clean.csv'),
                 encoding='utf-8-sig', dtype={'sound_id': 'string'})
df['tahun'] = pd.to_datetime(df['posted_at']).dt.year
urutan = ['<15 detik', '15-30 detik', '30-60 detik', '≥60 detik']
os.makedirs(os.path.join(BASE, 'images'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'outputs'), exist_ok=True)

def simpan(nama):
    plt.tight_layout()
    plt.savefig(os.path.join(BASE, 'images', nama), bbox_inches='tight', dpi=150)
    plt.show()

def spearman(a, b):
    m = a.notna() & b.notna()
    return round(a[m].rank().corr(b[m].rank()), 3)

print('Data:', df.shape, '| seed bootstrap: 42')


In [ ]:
# Sel 2: Seberapa presisi median? Bootstrap 2.000 kali
def bootstrap_median(x, n_boot=2000):
    x = np.asarray(x.dropna())
    boots = np.median(RNG.choice(x, size=(n_boot, len(x)), replace=True), axis=1)
    return np.percentile(boots, [2.5, 97.5])

lo, hi = bootstrap_median(df['like_rate'])
print(f'Median like_rate seluruh data: {df.like_rate.median()*100:.2f}% '
      f'(95% CI bootstrap: {lo*100:.2f}% - {hi*100:.2f}%)')

baris = []
for g in urutan:
    x = df.loc[df.duration_group == g, 'like_rate']
    l, h = bootstrap_median(x)
    baris.append({'durasi': g, 'n': len(x),
                  'median_pct': round(x.median()*100, 2),
                  'ci_bawah_pct': round(l*100, 2), 'ci_atas_pct': round(h*100, 2)})
boot = pd.DataFrame(baris)
print(boot.to_string(index=False))
boot.to_csv(os.path.join(BASE, 'outputs', '05_bootstrap_median.csv'),
            index=False, encoding='utf-8-sig')

plt.figure()
plt.errorbar(boot['median_pct'], boot['durasi'],
             xerr=[boot['median_pct'] - boot['ci_bawah_pct'],
                   boot['ci_atas_pct'] - boot['median_pct']],
             fmt='o', color=PALETTE[0], capsize=5)
plt.axvline(df.like_rate.median()*100, color=PALETTE[3], linestyle='--',
            label='Median seluruh data')
plt.title('Median Like Rate per Durasi dengan Interval Kepercayaan Bootstrap 95%')
plt.xlabel('Like rate median (%)')
plt.ylabel('Kelompok durasi')
plt.legend()
simpan('05_bootstrap_durasi.png')

bawah_1530 = boot.loc[boot.durasi == '15-30 detik', 'ci_bawah_pct'].iloc[0]
atas_60 = boot.loc[boot.durasi == '≥60 detik', 'ci_atas_pct'].iloc[0]
if bawah_1530 > atas_60:
    print('Interpretasi: interval 15-30 detik seluruhnya di atas interval ≥60 detik, '
          'jadi perbedaan itu sulit dijelaskan hanya oleh kebetulan sampel.')
else:
    print('Interpretasi: interval masih saling overlap, jadi perbedaan median bisa saja '
          'terjadi karena kebetulan sampel; klaim dilemahkan.')


In [ ]:
# Sel 3: Uji beda like rate antar kelompok durasi
grup = [df.loc[df.duration_group == g, 'like_rate'].values for g in urutan]
H, p = stats.kruskal(*grup)
print(f'Kruskal-Wallis: H = {H:.3f}, p = {p:.6g}')

pasangan = [('<15 detik', '15-30 detik'),
            ('15-30 detik', '≥60 detik'),
            ('<15 detik', '≥60 detik')]
baris = []
for a, b in pasangan:
    xa = df.loc[df.duration_group == a, 'like_rate']
    xb = df.loc[df.duration_group == b, 'like_rate']
    _, pv = stats.mannwhitneyu(xa, xb, alternative='two-sided')
    pc = min(pv * len(pasangan), 1.0)
    baris.append({'perbandingan': f'{a} vs {b}',
                  'p_mentah': round(pv, 6),
                  'p_bonferroni': round(pc, 6),
                  'signifikan_0.05': bool(pc < 0.05)})
uji = pd.DataFrame(baris)
print(uji.to_string(index=False))
uji.to_csv(os.path.join(BASE, 'outputs', '05_uji_durasi.csv'),
           index=False, encoding='utf-8-sig')

sig = uji.loc[uji.perbandingan == '15-30 detik vs ≥60 detik', 'signifikan_0.05'].iloc[0]
if sig:
    print('Interpretasi: pasangan 15-30 vs ≥60 detik tetap signifikan setelah koreksi '
          'Bonferroni, jadi pola durasi bukan sekadar fluktuasi acak.')
else:
    print('Interpretasi: pasangan 15-30 vs ≥60 detik tidak signifikan setelah koreksi, '
          'jadi pola durasi tidak cukup kuat untuk diklaim.')
print('Catatan: uji ini memakai peringkat (tahan outlier), bukan mean; signifikansi '
      'statistik tidak sama dengan pentingnya efek.')


In [ ]:
# Sel 4: Sensitivitas terhadap video viral - buang top 1% views
n_top = math.ceil(len(df) * 0.01)
top_idx = df.nlargest(n_top, 'views').index
tanpa = df.drop(top_idx)

def ringkas(d):
    return d.groupby('duration_group').agg(
        n=('video_id', 'size'),
        median_pct=('like_rate', lambda s: round(s.median()*100, 2))
    ).reindex(urutan)

sens = ringkas(df).join(ringkas(tanpa), lsuffix='_lengkap', rsuffix='_tanpa_top1')
sens['selisih_poin'] = (sens['median_pct_tanpa_top1'] -
                        sens['median_pct_lengkap']).round(2)
print(sens.to_string())
print('Spearman durasi vs like_rate - lengkap:', spearman(df.duration_sec, df.like_rate),
      '| tanpa top1%:', spearman(tanpa.duration_sec, tanpa.like_rate))
print('Top 1% menyumbang', round(df.loc[top_idx, 'views'].sum()/df.views.sum()*100, 1),
      '% total views')
sens.to_csv(os.path.join(BASE, 'outputs', '05_sensitivitas_top1pct.csv'),
            encoding='utf-8-sig')

plt.figure(figsize=(10, 5))
x = np.arange(len(urutan))
w = 0.38
plt.bar(x - w/2, sens['median_pct_lengkap'], w, label='Data lengkap', color=PALETTE[0])
plt.bar(x + w/2, sens['median_pct_tanpa_top1'], w, label='Tanpa top 1% views', color=PALETTE[1])
plt.xticks(x, urutan)
plt.title('Sensitivitas Like Rate Median: Data Lengkap vs Tanpa 1% Video Teratas')
plt.xlabel('Kelompok durasi')
plt.ylabel('Like rate median (%)')
plt.legend()
simpan('05_sensitivitas_top1pct.png')

maks = sens['selisih_poin'].abs().max()
if maks <= 1:
    print(f'Interpretasi: pengecualian {n_top} video viral mengubah median maksimal '
          f'{maks:.2f} poin persen, jadi temuan durasi tahan terhadap outlier.')
else:
    print(f'Interpretasi: pengecualian {n_top} video viral mengubah median sampai '
          f'{maks:.2f} poin persen, jadi temuan durasi sensitif terhadap video viral.')


In [ ]:
# Sel 5: Sensitivitas ambang "interaksi tinggi"
dummies = df['hashtags_clean'].str.get_dummies(sep=', ')

def top_lift(mask, lo_mask, min_atas=5):
    hi_idx = df.index[mask]
    lo_idx = df.index[lo_mask]
    rows = []
    for tag in dummies.columns:
        a = int(dummies.loc[hi_idx, tag].sum())
        b = int(dummies.loc[lo_idx, tag].sum())
        if a >= min_atas:
            lift = (a/len(hi_idx)) / ((b + 0.5)/len(lo_idx))
            rows.append({'item': tag, 'jenis': 'hashtag', 'n_atas': a,
                         'n_bawah': b, 'lift': round(lift, 2)})
    atas = df.loc[hi_idx, 'sound_id'].value_counts()
    bawah = df.loc[lo_idx, 'sound_id'].value_counts()
    for sid, a in atas.items():
        if a >= min_atas:
            b = int(bawah.get(sid, 0))
            lift = (a/len(hi_idx)) / ((b + 0.5)/len(lo_idx))
            rows.append({'item': str(sid), 'jenis': 'sound_id', 'n_atas': int(a),
                         'n_bawah': b, 'lift': round(lift, 2)})
    return pd.DataFrame(rows).sort_values('lift', ascending=False).head(5)

q25, q75 = df.like_rate.quantile([.25, .75])
q90 = df.like_rate.quantile(.90)
l25, l75 = df.likes.quantile([.25, .75])
# Pembanding selalu kuartal bawah, supaya yang berubah HANYA definisi grup atas.
definisi = {
    'q75 like_rate': (df.like_rate >= q75, df.like_rate <= q25),
    'q90 like_rate': (df.like_rate >= q90, df.like_rate <= q25),
    'q75 jumlah likes': (df.likes >= l75, df.likes <= l25),
}
semua = []
for nama, (mask, lo_mask) in definisi.items():
    top = top_lift(mask, lo_mask)
    print(f'=== {nama} (n atas = {int(mask.sum())}, n bawah = {int(lo_mask.sum())}) ===')
    print(top.to_string(index=False))
    semua.append(top.assign(definisi=nama))
lift_sens = pd.concat(semua, ignore_index=True)
lift_sens.to_csv(os.path.join(BASE, 'outputs', '05_sensitivitas_ambang.csv'),
                 index=False, encoding='utf-8-sig')

muncul = lift_sens.groupby('item')['definisi'].nunique()
stabil = sorted(muncul[muncul > 1].index)
print('Item yang muncul di lebih dari satu definisi:', stabil if stabil else 'tidak ada')
print('Interpretasi: item yang hanya muncul pada satu definisi tidak boleh dijadikan '
      'kesimpulan; item yang muncul di beberapa definisi lebih layak dibahas.')
print('Catatan: lift adalah asosiasi, bukan sebab-akibat.')


In [ ]:
# Sel 6: Apakah efek durasi hanya efek follower atau tahun?
bins = [0, 1e4, 1e5, 1e6, np.inf]
labels = ['<10 rb', '10 rb-100 rb', '100 rb-1 jt', '≥1 jt']
df['follower_tier'] = pd.cut(df.author_followers, bins=bins, labels=labels, right=False)

print('=== Like rate median di dalam tiap tier follower (%) ===')
cek = []
for t in labels:
    sub = df[df.follower_tier == t]
    med = (sub.groupby('duration_group')['like_rate'].median() * 100).round(2)
    a, b = med.get('15-30 detik', np.nan), med.get('≥60 detik', np.nan)
    cek.append((t, len(sub), a, b))
    print(f'{t:14s} n={len(sub):3d} | 15-30 dtk: {a} | ≥60 dtk: {b}')

valid = [(t, a, b) for t, n, a, b in cek if pd.notna(a) and pd.notna(b)]
if valid and all(a > b for _, a, b in valid):
    print('Interpretasi: 15-30 detik lebih tinggi dari ≥60 detik di SEMUA tier follower, '
          'jadi pola ini bukan murni efek ukuran akun.')
else:
    print('Interpretasi: pola durasi tidak konsisten di semua tier follower, jadi efek '
          'follower perlu diwaspadai.')

tahun = df.groupby('tahun').agg(
    n=('video_id', 'size'),
    median_like_pct=('like_rate', lambda s: round(s.median()*100, 2))
)
print()
print('=== Like rate median per tahun ===')
print(tahun.to_string())
print('Spearman tahun vs like_rate:', spearman(df.tahun, df.like_rate))
print('Interpretasi: video lama punya waktu lebih lama untuk mengumpulkan views; '
      'karena itu perbandingan memakai like_rate (rasio), bukan jumlah likes mentah.')

print()
print('=== Cek durasi DI DALAM tiap tahun (min 10 video per grup) ===')
cek_tahun = []
for y, sub in df.groupby('tahun'):
    a_sub = sub.loc[sub.duration_group == '15-30 detik', 'like_rate']
    b_sub = sub.loc[sub.duration_group == '≥60 detik', 'like_rate']
    if len(a_sub) >= 10 and len(b_sub) >= 10:
        a, b = round(a_sub.median()*100, 2), round(b_sub.median()*100, 2)
        cek_tahun.append((int(y), a, b))
        print(f'{y}: 15-30 dtk n={len(a_sub):2d} {a}% | ≥60 dtk n={len(b_sub):2d} {b}%')
if cek_tahun:
    menang = sum(1 for _, a, b in cek_tahun if a > b)
    if menang == len(cek_tahun):
        print('Interpretasi: pola durasi tetap sama di semua tahun yang cukup sampel.')
    else:
        print(f'Interpretasi: dari {len(cek_tahun)} tahun yang cukup sampel, hanya {menang} '
              'yang menunjukkan 15-30 detik lebih tinggi; efek tahun perlu diwaspadai '
              'dan klaim durasi tidak boleh dibebaskan dari keterangan ini.')


In [ ]:
# Sel 7: Ringkasan ketahanan temuan
ringkasan = pd.DataFrame([
    {'temuan': 'Median like_rate seluruh data',
     'hasil': f'{df.like_rate.median()*100:.2f}% (CI {lo*100:.2f}% - {hi*100:.2f}%)',
     'status': 'Presisi'},
    {'temuan': '15-30 detik vs ≥60 detik',
     'hasil': f'Bonferroni signifikan: {bool(sig)}',
     'status': 'Uji peringkat'},
    {'temuan': 'Tahan terhadap pengecualian top 1%',
     'hasil': f'Perubahan maksimal {maks:.2f} poin persen',
     'status': 'Sensitivitas outlier'},
    {'temuan': 'Item interaksi tinggi stabil',
     'hasil': ', '.join(stabil) if stabil else 'tidak ada yang stabil',
     'status': 'Sensitivitas ambang'},
    {'temuan': 'Efek durasi bukan efek follower',
     'hasil': 'Konsisten di semua tier' if (valid and all(a > b for _, a, b in valid))
              else 'Tidak konsisten',
     'status': 'Kontrol variabel'},
    {'temuan': 'Efek durasi stabil antar tahun',
     'hasil': f'{sum(1 for _, a, b in cek_tahun if a > b)} dari {len(cek_tahun)} tahun'
              if cek_tahun else 'tidak ada tahun dengan sampel cukup',
     'status': 'Kontrol tahun'},
])
print(ringkasan.to_string(index=False))
ringkasan.to_csv(os.path.join(BASE, 'outputs', '05_ringkasan_sensitivitas.csv'),
                 index=False, encoding='utf-8-sig')
print('Simpan:', '05_ringkasan_sensitivitas.csv')


## Cara menjawab di interview

- **Bootstrap** dipakai karena distribusi like_rate miring; kita tidak bisa memakai asumsi normal untuk mean.
- **Kruskal-Wallis + Mann-Whitney** dipakai karena ini uji berbasis peringkat, jadi tidak terpengaruh video viral.
- **Bonferroni** dipakai karena tiga pasangan diuji sekaligus; tanpa koreksi, risiko salah klaim naik.
- **Sensitivitas** dijalankan dengan sengaja: kalau kesimpulan berubah hanya karena satu pilihan metode, kesimpulan itu tidak layak dipakai.
- Tetap jujur: signifikan statistik ≠ penting secara bisnis, dan asosiasi ≠ sebab-akibat.
